# Flow Matching: Equality Constraints (Lagrangian)

Solve the gasoline blending problem using Flow Matching with the Lagrangian method.

**Problem:** Find the blend closest to a target recipe $(0.4, 0.4, 0.2)$ while meeting octane and mass balance constraints.

We use the Lagrangian method: condition on all partial derivatives being zero.

**Authors:** Victor Alves and John R. Kitchin

## System Information

This section documents the computational environment used to run this notebook.

In [1]:
import sys
import platform
import time
import psutil
import os

print("System Information")
print("=" * 70)
print(f"Python version: {sys.version}")
print(f"Platform: {platform.platform()}")
print(f"Processor: {platform.processor()}")
print(f"CPU count: {psutil.cpu_count(logical=False)} physical, {psutil.cpu_count(logical=True)} logical")
print(f"Total memory: {psutil.virtual_memory().total / (1024**3):.2f} GB")
print("=" * 70)

# Key package versions
print("\nKey Library Versions:")
print("-" * 70)
packages = [
    ('numpy', 'np'),
    ('scipy', 'scipy'),
    ('matplotlib', 'matplotlib'),
    ('sklearn', 'sklearn'),
    ('torch', 'torch'),
    ('gmr', 'gmr'),
]

for pkg_name, import_name in packages:
    try:
        mod = __import__(pkg_name)
        version = getattr(mod, '__version__', 'unknown')
        print(f"{pkg_name:<20} {version}")
    except ImportError:
        print(f"{pkg_name:<20} NOT INSTALLED")

print("-" * 70)

# Initialize timing and memory tracking
_start_time = time.time()
_process = psutil.Process(os.getpid())
_start_memory_mb = _process.memory_info().rss / (1024 * 1024)

print(f"\nNotebook execution started: {time.strftime('%Y-%m-%d %H:%M:%S', time.localtime(_start_time))}")
print(f"Initial memory usage: {_start_memory_mb:.2f} MB")
print("=" * 70)

System Information
Python version: 3.12.8 (main, Jan 14 2025, 23:36:58) [Clang 19.1.6 ]
Platform: macOS-15.7.2-arm64-arm-64bit
Processor: arm
CPU count: 14 physical, 14 logical
Total memory: 64.00 GB

Key Library Versions:
----------------------------------------------------------------------
numpy                2.3.4
scipy                1.16.2
matplotlib           3.10.1


sklearn              1.7.2


torch                2.4.1
gmr                  2.0.2
----------------------------------------------------------------------

Notebook execution started: 2025-12-18 14:24:26
Initial memory usage: 317.92 MB


## System Information

This section documents the computational environment used to run this notebook.

In [2]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
import warnings
import os

# Force CPU for JAX (must be set before importing JAX)
os.environ['JAX_PLATFORMS'] = 'cpu'

import torch
import jax
import jax.numpy as jnp
from jax import jacobian, vmap
from scipy.optimize import minimize

# Import reusable utilities from local module
from generative_optimization import (
    generate_samples,
    cluster_stats,
    ConditionalFlowMatching
)

# Force CPU for PyTorch
device = torch.device('cpu')
print(f"Using device: {device}")

# Figure settings
mpl.rcParams['figure.facecolor'] = 'white'
mpl.rcParams['axes.facecolor'] = 'white'
mpl.rcParams['figure.dpi'] = 150

warnings.filterwarnings('ignore')

# Set seeds for reproducibility
torch.manual_seed(42)
np.random.seed(42)

Using device: cpu


## Problem Setup

**Objective:** Minimize squared distance from target blend $(x_1^*, x_2^*, x_3^*) = (0.4, 0.4, 0.2)$
$$\min_{x_1, x_2, x_3} \quad (x_1 - 0.4)^2 + (x_2 - 0.4)^2 + (x_3 - 0.2)^2$$

**Constraints:**
- Octane: $95 x_1 + 82 x_2 + 94 x_3 = 87$
- Mass balance: $x_1 + x_2 + x_3 = 1$

The Lagrangian is:
$$L(x_1, x_2, x_3, \lambda_1, \lambda_2) = \sum_i (x_i - x_i^*)^2 + \lambda_1(95x_1 + 82x_2 + 94x_3 - 87) + \lambda_2(x_1 + x_2 + x_3 - 1)$$

At the optimum, all partial derivatives are zero:
- $\partial L/\partial x_1 = 2(x_1 - 0.4) + 95\lambda_1 + \lambda_2 = 0$
- $\partial L/\partial x_2 = 2(x_2 - 0.4) + 82\lambda_1 + \lambda_2 = 0$
- $\partial L/\partial x_3 = 2(x_3 - 0.2) + 94\lambda_1 + \lambda_2 = 0$
- $\partial L/\partial \lambda_1 = 95x_1 + 82x_2 + 94x_3 - 87 = 0$
- $\partial L/\partial \lambda_2 = x_1 + x_2 + x_3 - 1 = 0$

In [3]:
# Scipy solution for comparison
x_target = np.array([0.4, 0.4, 0.2])
rons = np.array([95.0, 82.0, 94.0])
ron_target = 87.0

def objective(x):
    return np.sum((x - x_target)**2)

def octane_constraint(x):
    return rons @ x - ron_target

def mass_balance(x):
    return np.sum(x) - 1

sol = minimize(objective, [0.33, 0.33, 0.34], 
               constraints=[
                   {'type': 'eq', 'fun': octane_constraint},
                   {'type': 'eq', 'fun': mass_balance}
               ])
print(f"Scipy solution: x1={sol.x[0]:.4f}, x2={sol.x[1]:.4f}, x3={sol.x[2]:.4f}")
print(f"Objective: {sol.fun:.6f}")

Scipy solution: x1=0.2841, x2=0.6070, x3=0.1089
Objective: 0.064586


In [4]:
# Generate data with Lagrangian derivatives
x_target_jax = jnp.array([0.4, 0.4, 0.2])
rons_jax = jnp.array([95.0, 82.0, 94.0])

def L(Y):
    """Lagrangian: L = objective + lambda1 * g1 + lambda2 * g2"""
    x = Y[0:3]
    lam1, lam2 = Y[3], Y[4]
    # Objective: squared distance from target
    obj = jnp.sum((x - x_target_jax)**2)
    # Constraints
    g1 = jnp.dot(rons_jax, x) - 87.0  # Octane constraint
    g2 = jnp.sum(x) - 1.0              # Mass balance
    return obj + lam1 * g1 + lam2 * g2

# Sample [x1, x2, x3, lambda1, lambda2]
# λ2 ≈ -4.5 at the solution
bounds = [[0, 1], [0, 1], [0, 1], [-0.5, 0.5], [-6, 0]]
Y = generate_samples(bounds, n_samples=512, seed=42)

# Compute Lagrangian derivatives
dL = vmap(jacobian(L))(Y)
dL = np.array(dL)

print(f"Y shape: {Y.shape}")
print(f"dL shape: {dL.shape}")

Y shape: (512, 5)
dL shape: (512, 5)


In [5]:
# Train: generate [x1, x2, x3, lambda1, lambda2] conditioned on dL = 0
x_data = Y  # [x1, x2, x3, lambda1, lambda2]
c_data = dL  # [dL/dx1, dL/dx2, dL/dx3, dL/dlambda1, dL/dlambda2]

fm_lagrange = ConditionalFlowMatching(x_dim=5, c_dim=5, hidden_dim=128, n_layers=4)
losses = fm_lagrange.fit(x_data, c_data, epochs=1000, batch_size=64)

Training:   0%|          | 0/1000 [00:00<?, ?it/s]

In [6]:
# Sample by conditioning on all derivatives = 0
samples = fm_lagrange.sample(c_values=[[0.0, 0.0, 0.0, 0.0, 0.0]], n_samples=500)

# Get mean solution
x_opt = samples.mean(axis=0)
print(f"Flow Matching solution:")
print(f"  x1 (Reformate) = {x_opt[0]:.4f}")
print(f"  x2 (Isomerate) = {x_opt[1]:.4f}")
print(f"  x3 (Alkylate)  = {x_opt[2]:.4f}")
print(f"  λ1 = {x_opt[3]:.4f}")
print(f"  λ2 = {x_opt[4]:.4f}")
print(f"\nExpected: x1=0.284, x2=0.607, x3=0.109")

Flow Matching solution:
  x1 (Reformate) = 0.2686
  x2 (Isomerate) = 0.6001
  x3 (Alkylate)  = 0.1311
  λ1 = 0.0263
  λ2 = -2.9939

Expected: x1=0.284, x2=0.607, x3=0.109


In [7]:
# Check constraint satisfaction
octane_vals = 95*samples[:, 0] + 82*samples[:, 1] + 94*samples[:, 2] - 87
mass_vals = samples[:, 0] + samples[:, 1] + samples[:, 2] - 1
print(f"Octane constraint violation (mean |error|): {np.mean(np.abs(octane_vals)):.6f}")
print(f"Mass balance violation (mean |error|): {np.mean(np.abs(mass_vals)):.6f}")

# Objective value
x_mean = samples[:, 0:3].mean(axis=0)
obj_fm = np.sum((x_mean - x_target)**2)
print(f"\nObjective (distance²): {obj_fm:.6f}")
print(f"Scipy objective: {sol.fun:.6f}")

Octane constraint violation (mean |error|): 0.512274
Mass balance violation (mean |error|): 0.005614

Objective (distance²): 0.062046
Scipy objective: 0.064586


In [8]:
# Final timing and memory report
_end_time = time.time()
_end_memory_mb = _process.memory_info().rss / (1024 * 1024)
_elapsed_seconds = _end_time - _start_time

print("\n" + "=" * 70)
print("NOTEBOOK EXECUTION SUMMARY")
print("=" * 70)
print(f"Start time:      {time.strftime('%Y-%m-%d %H:%M:%S', time.localtime(_start_time))}")
print(f"End time:        {time.strftime('%Y-%m-%d %H:%M:%S', time.localtime(_end_time))}")
print(f"Elapsed time:    {_elapsed_seconds:.2f} seconds ({_elapsed_seconds/60:.2f} minutes)")
print("-" * 70)
print(f"Initial memory:  {_start_memory_mb:.2f} MB")
print(f"Final memory:    {_end_memory_mb:.2f} MB")
print(f"Memory change:   {_end_memory_mb - _start_memory_mb:+.2f} MB")
print("=" * 70)


NOTEBOOK EXECUTION SUMMARY
Start time:      2025-12-18 14:24:26
End time:        2025-12-18 14:24:38
Elapsed time:    11.87 seconds (0.20 minutes)
----------------------------------------------------------------------
Initial memory:  317.92 MB
Final memory:    579.30 MB
Memory change:   +261.38 MB
